In [1]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
import agentops
import os
from pydantic import BaseModel, Field
from typing import List, Optional
from tavily import TavilyClient
from scrapegraph_py import ScrapeGraphAI
import json

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
AGENTOPS_API_KEY = os.getenv("AGENTOPS_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
SGAI_API_KEY = os.getenv("SGAI_API_KEY")

In [2]:
output_dir = "./ai_agent_output"
os.makedirs(output_dir, exist_ok=True)

In [3]:
basic_llm = LLM(
    model="gpt-4o",
    temperature=0
    )
agentops.init(api_key=AGENTOPS_API_KEY, skip_auto_end_session=True)
search_client = TavilyClient(api_key=TAVILY_API_KEY)
scrape_client = ScrapeGraphAI(api_key=SGAI_API_KEY)


🖇 AgentOps: You're on the agentops free plan 🤔


In [4]:
no_keywords=10

## Setup Agents

### 1. Search Queries Recommendation Agent

In [5]:
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(..., title="A list of suggested search queries.", min_items=1, max_items=no_keywords)

search_queries_recommendation_agent = Agent(
    role="Search Queries Recommendation Agent",
    goal="\n".join([
        "To provide a list of suggested search queries to be passed to the search engine.",
        "The queries must be varied and looking for specific items."
    ]),
    backstory="The agent is designted to help looking for products by providing a list of suggested search queries to be passed to the search engine.",
    llm=basic_llm,
    verbose=True
)

search_queries_recommendation_task = Task(
    description="\n".join([
        "Storey is looking to buy {product_name} at the best prices (value for a price strategy)",
        "The company target any of these websites to buy from: {websites_list}",
        "The company wants to reach all available prodcuts on the internet to be compared later in another stage.",
        "The stores must sell the product in {country_name}",
        "Generate maximum {no_keywords} queries.",
        "The search query must reach an ecommerce webpage for product, and not a blog or listing page."
    ]), 
    expected_output="A JSON object containing a list of seggested search queries.",
    output_json=SuggestedSearchQueries,
    output_file=os.path.join(output_dir, "step_1_suggested_search_queries.json"),
    agent=search_queries_recommendation_agent
)

C:\Temp\ipykernel_20144\2998558793.py:2: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  queries: List[str] = Field(..., title="A list of suggested search queries.", min_items=1, max_items=no_keywords)
C:\Temp\ipykernel_20144\2998558793.py:2: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  queries: List[str] = Field(..., title="A list of suggested search queries.", min_items=1, max_items=no_keywords)


### 2. Search Engine Agent

in this agent we want to use a tool for searching

so we will use tavily to make our own tool

This is the output shape, so make a pydantic like it.

In [ ]:
{
    "results": [
        {
            "title":"",
            "url":"",
            "content":"",
            "score":"",
            "search_query":""
        }
    ]
}

In [6]:
@tool
def search_engine_tool(query:str):
    # for tools, we should write this comment 
    """Useful for based search quries. Use This to find current information about any related pages using a search engine."""
    return search_client.search(query=query)

class SingleSearchResult(BaseModel):
    title: str = Field(..., title="The title of the search result.")
    url: str = Field(..., title="The URL of the search result.")
    content: str = Field(..., title="The content of the search result.")
    score: float = Field(..., title="The confidence score of the search result.")
    search_query: str = Field(..., title="The search query that generated this result.")

class AllSearchResults(BaseModel):
    results: List[SingleSearchResult] = Field(..., title="A list of search results.")

search_engine_agent= Agent(
    role="Search Engine Agent",
    goal="To search for products based on the suggested search quries.",
    backstory="The agent is designed to help looking for products by searching for the suggested search queries.",
    llm=basic_llm,
    verbose=True,
    tools=[search_engine_tool]
)

search_engine_task = Task(
    description="\n".join([
        "The task is to search for products based on the suggested search queries.",
        "You have to collect results from multiple search queries.",
        "Ignore any suspicious links or not an ecommerce single product website link.",
        "Ignore any search results with confidence score less than ({score_th}).",
        "The search results will be used to compare prices of products from different websites.",
    ]),
    expected_output="A JSON object containing the search results.",
    output_json=AllSearchResults,
    output_file=os.path.join(output_dir, "step_2_search_results.json"),
    agent=search_engine_agent
)

### 3. Scraping Agent

In [ ]:
{
    "products": [
        {
            "page_url": "",
            "product_title": "",
            "product_url": "",
            "product_image_url": "",
            "product_current_price": "",
            "product_original_price": "",
            "product_discount_percentage": "",
        }
    ]
}

In [7]:
class SingleProductInfo(BaseModel):
    page_url: str
    product_title: str
    product_url: str
    product_image_url: str
    product_current_price: str
    product_original_price: str
    product_discount_percentage: str
    agent_recommendation_rank: int = Field(..., title="The rank of the product to be considered in the final procurement report. (out of 5, Higher is Better) in the recommendation list ordering from the best to the worst")
    agent_recommendation_notes: List[str]  = Field(..., title="A set of notes why would you recommend or not recommend this product to the company, compared to other products.")

class AllProductsInfo(BaseModel):
    products: List[SingleProductInfo] = Field(..., title="A list of products information.")


@tool
def website_scraping_tool(url:str):
    # for tools, we should write this comment 
    """Useful for scraping product information from the search results. Use This to scrape the product information from the search results."""

    details= scrape_client.extract(
    prompt= "Extract ```json\n"+  SingleProductInfo.schema_json()+ "```\n from the webpage.",
    url=url,
    )
    return {
        "page_url":url,
        "details":details
    }

scraping_agent= Agent(
    role="Scraping Agent",
    goal="To scrape the product information from the search results.",
    backstory="The agent is designed to help looking for products by scraping the product information from the search results.",
    llm=basic_llm,
    verbose=True,
    tools=[website_scraping_tool]
)

scraping_task= Task(
    description="\n".join([
        "The task is to scrape the product information from the search results.",
        "You have to collect product information from multiple search results.",
        "The product information will be used to compare prices of products from different websites.",
        "Collect the best {top_recommendations_no} products from the search results.",
    ]),
    expected_output="A JSON object containing the product information.",
    output_json=AllProductsInfo,
    output_file=os.path.join(output_dir, "step_3_scraped_products_info.json"),
    agent=scraping_agent
)

### 4. Report Agent

In [8]:
report_agent= Agent(
    role="Report Agent",
    goal="To generate a professional HTML page for the procurement report.",
    backstory="The agent is designed to help generate professional HTML page for procurement reports.",
    llm=basic_llm,
    verbose=True
)

report_task= Task(
    description="\n".join([
    "The task is to generate a professional HTML page for the procurement report.",
    "You have to use Bootstrap CSS framework for a better UI.",
    "The report will include the search results and prices of products from different websites.",
    "The report should be structured with the following sections:",
    "1. Executive Summary: A brief overview of the procurement process and key findings.",
    "2. Introduction: An introduction to the purpose and scope of the report.",
    "3. Methodology: A description of the methods used to gather and compare prices.",
    "4. Findings: Detailed comparison of prices from different websites, including tables and charts.",
    "5. Analysis: An analysis of the findings, highlighting any significant trends or observations.",
    "6. Recommendations: Suggestions for procurement based on the analysis.",
    "7. Conclusion: A summary of the report and final thoughts.",
    "8. Appendices: Any additional information, such as raw data or supplementary materials.",
    ]),
    expected_output="A HTML page containing the procurement report.",
    output_file=os.path.join(output_dir, "step_4_procurement_report.html"),
    agent=report_agent
)

## Run Crew

In [9]:
storey_crew = Crew(
    agents=[search_queries_recommendation_agent, search_engine_agent, scraping_agent, report_agent],
    tasks=[search_queries_recommendation_task, search_engine_task, scraping_task, report_task],
    process=Process.sequential
)


In [ ]:
"""if in google colab:
crew_results= storey_crew.kickoff(
    inputs={
    "product_name": "coffee machine for the office",
    "websites_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
    "country_name": "Egypt",
    "no_keywords": 10,
}
)
"""

In [ ]:
crew_results= await storey_crew.kickoff_async(
    inputs={
    "product_name": "coffee machine for the office",
    "websites_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
    "country_name": "Egypt",
    "no_keywords": 10,
    "score_th": 0.10,
    "top_recommendations_no": 10
}
) 